In [0]:
%sql

-- Build the final hourly inference dataset used for model prediction.
--
-- This table combines:
-- - consumption-based lag and rolling features
-- - group-level metadata
-- - group-to-weather mapping
-- - lagged electricity price features
-- - lagged weather features that contain null values
-- - calendar features
--
-- Join logic:
-- - consumption is the base grain of the dataset:
--   one row per `group_id` and `timestamp_utc`
-- - group metadata is joined by `group_id`
-- - weather mapping is joined by `group_id`
-- - price and calendar are joined by `timestamp_utc`
-- - weather is joined by both `timestamp_utc` and mapped `weather_key`
--
-- Final output grain:
-- - one row per `group_id` and `timestamp_utc`

CREATE OR REPLACE TABLE fortum_challenge_data.03_gold.hourly_inference_dataset_weather_null AS

WITH consumption AS (
    SELECT
        timestamp_utc,
        group_id,
        target_consumption,
        consumption_lag_48,
        consumption_lag_168,
        consumption_lag_336,
        rolling_mean_168,
        rolling_std_168,
        lag_diff_48_168,
        lag_diff_168_336
    FROM fortum_challenge_data.02_silver.consumption_hourly_inference_clean
),

group_data AS (
    SELECT
        group_id,
        macro_region,
        region,
        municipality,
        segment,
        product_type,
        consumption_bucket
    FROM fortum_challenge_data.02_silver.group_data_clean
),

group_weather_mapping AS (
    SELECT
        group_id,
        weather_key
    FROM fortum_challenge_data.02_silver.group_weather_map
),

price AS (
    SELECT
        timestamp_utc,
        price_lag_24,
        price_lag_48,
        price_lag_168
    FROM fortum_challenge_data.02_silver.price_inference_clean
),

weather AS (
    SELECT
        timestamp_utc,
        weather_key,
        temperature_missing_flag,
        temp_lag_24,
        temp_lag_48,
        temp_lag_168
    FROM fortum_challenge_data.02_silver.weather_inference_null_clean
),

calendar AS (
    SELECT
        timestamp_utc,
        is_weekend,
        is_holiday,
        working_day,
        desc,
        hour_of_day,
        day_of_week,
        month,
        day_of_year
    FROM fortum_challenge_data.02_silver.calendar_hourly_inference_clean
),

joined AS (
    SELECT
        c.timestamp_utc,
        c.group_id,
        c.target_consumption,
        c.consumption_lag_48,
        c.consumption_lag_168,
        c.consumption_lag_336,
        c.rolling_mean_168,
        c.rolling_std_168,
        c.lag_diff_48_168,
        c.lag_diff_168_336,
        g.macro_region,
        g.region,
        g.municipality,
        g.segment,
        g.product_type,
        g.consumption_bucket,
        gwm.weather_key,
        p.price_lag_24,
        p.price_lag_48,
        p.price_lag_168,
        w.temperature_missing_flag,
        w.temp_lag_24,
        w.temp_lag_48,
        w.temp_lag_168,
        cal.working_day,
        cal.desc,
        cal.is_weekend,
        cal.is_holiday,
        cal.hour_of_day,
        cal.day_of_week,
        cal.month,
        cal.day_of_year
    FROM consumption c
    LEFT JOIN group_data g
        ON c.group_id = g.group_id
    LEFT JOIN group_weather_mapping gwm
        ON c.group_id = gwm.group_id
    LEFT JOIN price p
        ON c.timestamp_utc = p.timestamp_utc
    LEFT JOIN weather w
        ON c.timestamp_utc = w.timestamp_utc
       AND gwm.weather_key = w.weather_key
    LEFT JOIN calendar cal
        ON c.timestamp_utc = cal.timestamp_utc
)

SELECT *
FROM joined
ORDER BY group_id, timestamp_utc;